# Data Cleaning

**Roadmap:** Finding blanks → Deleting → Filling → Extreme values → ☕ → Transforming → Duplicates → Saving → 🚢 Titanic

📂 **Datasets:** `planets.csv` and `Titanic-Dataset.csv` — keep both in this folder.

In [ ]:
import numpy as np 
import pandas as pd

In [ ]:
df=pd.read_csv("/Users/shailesh/Desktop/vedam_Sem3_ml/Data/Titanic-Dataset.csv")
p=pd.read_csv("/Users/shailesh/Desktop/vedam_Sem3_ml/Data/planets.csv")

# 🔁 Revision — Day 4

Same file as last class. Nothing new here — every question uses something
you already wrote in last class.



**1.** The archive wants a list of the oldest people on board.
Show only `Name`, `Pclass` and `Fare` for passengers older than 65.
How many are there?

**2.** How many passengers did **not** board at Southampton? Use `~`.




In [ ]:
df.head(1)

**3.** How many passengers were either under 10 or over 70? Use `|`.

**4.** How many passengers travelled in each class?

Then, among **third class only**, how many were male and how many female?

**5.** Create a column called `travel_group`. Everyone starts as `'alone'`.
Anyone whose `SibSp` plus `Parch` is more than 0 becomes `'with family'`.

In [ ]:
df.head(1)
df[(df['Age']<10) |(df["Age"]>70)].shape[0]
df["Pclass"].value_counts()
df['travel_group']="alone"
df.loc[(df['SibSp']+df['Parch']>0),'travel_group']="With faimily"

In [ ]:
df['travel_group'].value_counts()

### Yesterday

150 passengers over 40, 564 aged 40 or under. Total **714**, not 891.
177 rows were dropped from every answer and nothing warned us.

Three options exist, and each one changes the answer: **delete**, **fill**, or **leave it**.
Today we do all three.

In [ ]:
p.head()

**1035 planets discovered outside our solar system, 1989–2014.**

| Column | Meaning |
|---|---|
| `method` | how it was detected |
| `number` | planets known in that star system |
| `orbital_period` | days to orbit once |
| `mass` | in Jupiter masses |
| `distance` | from Earth, in parsecs |
| `year` | year of discovery |

---
## 1️⃣ Finding blanks

> **Flow:** one cell → one column → the whole table

In [ ]:
# True wherever a value is missing
p.isnull().sum()

In [ ]:
# count the blanks in one column
print(p['mass'].isnull().sum())

In [ ]:
# same information, different view


| Column | Blanks |
|---|---|
| `mass` | 522 |
| `distance` | 227 |
| `orbital_period` | 43 |
| `method`, `number`, `year` | 0 |

`isnull()` and `isna()` are the same function — two names for one thing.
`notnull()` is its opposite.

### ✋ Guess first

`p.dropna()` removes every row with any blank in it. We have **1035** planets.





In [ ]:
p.dropna()

**498 of 1035.** Half the catalogue, and most of it went because of `mass` —
a column you may not even have needed.

In [ ]:
# drop only where mass is blank
p.dropna(subset=['mass'])


In [ ]:
# drop the damaged columns instead, and see which survive
p.dropna(axis=1)

| Command | Result |
|---|---|
| `p.dropna()` | 498 rows |
| `p.dropna(subset=['mass'])` | 513 rows |
| `p.dropna(subset=['distance'])` | 808 rows |
| `p.dropna(axis=1)` | 3 columns — only `method`, `number`, `year` |

⚠️ No `subset` means **any** blank anywhere kills the row.

### 🧩 Try these — on the **Titanic** file


**2.1** Two Titanic passengers have no boarding port recorded. Drop only those rows.
How many passengers remain?


In [ ]:
df.dropna(subset=['Embarked'])

**2.2** Keep all 891 passengers, but drop every **column** that has a blank.
How many columns survive, and which three did you lose?

In [66]:
df.dropna(axis=1)

,PassengerId,Survived,Pclass,Name,Sex,SibSp,Parch,Ticket,Fare,travel_group
0,1,0,3,"Braund, Mr. Owen Harris",male,1,0,A/5 21171,7.2500,With faimily
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,1,0,PC 17599,71.2833,With faimily
2,3,1,3,"Heikkinen, Miss. Laina",female,0,0,STON/O2. 3101282,7.9250,alone
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,1,0,113803,53.1000,With faimily
4,5,0,3,"Allen, Mr. William Henry",male,0,0,373450,8.0500,alone
...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,0,0,211536,13.0000,alone
887,888,1,1,"Graham, Miss. Margaret Edith",female,0,0,112053,30.0000,alone
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,1,2,W./C. 6607,23.4500,With faimily
889,890,1,1,"Behr, Mr. Karl Howell",male,0,0,111369,30.0000,alone


---
## 3️⃣ Filling

```python
p['mass'].fillna(0)                    # a fixed value
p['mass'].fillna(p['mass'].mean())     # the average
p['mass'].fillna(p['mass'].median())   # the middle value
```

> **Flow:** mean vs median → fill with each → what the fill did to the data

In [ ]:
# mean and median of mass
p['mass'].fillna(0).isnull().sum()

np.int64(0)

**Mean 2.64, median 1.26.** The mean is double the median — a few very heavy planets pull it up.

In [ ]:
# fill mass with the mean, then check mean / median / std

| | before | after mean-fill |
|---|---|---|
| mean | 2.64 | 2.64 |
| **median** | **1.26** | **2.64** |
| **std** | **3.82** | **2.69** |

522 identical values landed in the middle of the column. The median moved to sit on top of them
and the spread shrank.

### 🧩 Try these — on the **Titanic** file

The demo above was on `planets`. These are the same moves on `t`, so you are
practising the idea and not repeating the cells.

**3.1** Work on a copy first: `t2 = t.copy()`

Fill the missing `Age` in `t2` with the **mean**, then count how many passengers are now
recorded as exactly **29.6991** years old. Before you ran it, that count was **0**.

In [ ]:
# 3.1
# t2 = t.copy()   <- do not touch t, we need its blanks later

**3.2** `Fare` has mean **32.20** and median **14.45** — more than double apart.

If `Fare` had blanks, which of the two would you fill with? One line, no code.

In [ ]:
# 3.2

---
## 5️⃣ Duplicates

```python
p.duplicated()                        # True for a row that repeats an earlier one
p.duplicated().sum()                  # how many
p.drop_duplicates()                   # remove them
p.duplicated(subset=['method'])       # "same" only in the columns you name
```

> **Flow:** find → count → show → remove → then decide what "the same" means

In [71]:
# how many exact duplicate rows?
df.duplicated().sum()
p.duplicated().sum()

np.int64(4)

In [72]:
# show them
p.drop_duplicates()

,method,number,orbital_period,mass,distance,year
0,Radial Velocity,1,269.300000,7.10,77.40,2006
1,Radial Velocity,1,874.774000,2.21,56.95,2008
2,Radial Velocity,1,763.000000,2.60,19.84,2011
3,Radial Velocity,1,326.030000,19.40,110.62,2007
4,Radial Velocity,1,516.220000,10.50,119.47,2009
...,...,...,...,...,...,...
1030,Transit,1,3.941507,NaN,172.00,2006
1031,Transit,1,2.615864,NaN,148.00,2007
1032,Transit,1,3.191524,NaN,174.00,2007
1033,Transit,1,4.125083,NaN,293.00,2008


In [77]:
p.drop_duplicates(subset=['method'],ignore_index=True)

,method,number,orbital_period,mass,distance,year
0,Radial Velocity,1,269.300000,7.10,77.40,2006
1,Imaging,1,NaN,NaN,45.52,2005
2,Eclipse Timing Variations,1,10220.000000,6.05,NaN,2009
3,Transit,1,1.508956,NaN,NaN,2008
4,Astrometry,1,246.360000,NaN,20.77,2013
5,Transit Timing Variations,2,160.000000,NaN,2119.00,2011
6,Orbital Brightness Modulation,2,0.240104,NaN,1180.00,2011
7,Microlensing,1,NaN,NaN,NaN,2008
8,Pulsar Timing,3,25.262000,NaN,NaN,1992
9,Pulsation Timing Variations,1,1170.000000,NaN,NaN,2007


In [ ]:
# remove them and check the row count

**4 exact duplicates → 1031 rows.** Two Imaging pairs and two Microlensing pairs.

⚠️ `subset=` changes the whole meaning:

In [ ]:
# duplicates by method only

### 🧩 Try this — on the **Titanic** file

**B.1** Save the Titanic table as `titanic_clean.csv` with `index=False`,
read it back into a new variable, and confirm the columns are identical to `t`.

In [ ]:
# B.1

**T1** Which three columns are damaged, and by how much?

In [ ]:
# T1

**T2** Run `t.dropna()` with no arguments. How many of the 891 passengers survive?
Which column caused most of that damage?

**T3** Fill `Embarked` with the most common port, and `Age` with the median.

**T4** Build a `has_cabin` column and compare the survival rate of the two groups.

You did the same move on `Age` in 4.1. Same idea, different column — watch what changes.

**66.7% against 30.0%.** The most damaged column in the file — 687 blanks out of 891 —
separates survivors better than almost anything else in it. A recorded cabin number meant
first class, upper deck, near the boats.